In [ ]:
import pandas as pd
from glob import glob
import sys
import os
import sklearn.neighbors._base
sys.modules['sklearn.neighbors.base'] = sklearn.neighbors._base

from tqdm.notebook import tqdm

from missingpy import MissForest

In [ ]:
for file in tqdm(glob("*_0515.csv")):
    dataset_until_15 = pd.read_csv(file, sep=";", decimal=",", parse_dates=["FECHA_HORA"], index_col="FECHA_HORA")
    dataset_after_15 = pd.read_csv(file.split("_")[0]+"_1523.csv", sep=";", decimal=",", parse_dates=["FECHA_HORA"], index_col="FECHA_HORA")
    dataset_after_15 = dataset_after_15[(dataset_after_15.index.year>2015) & (dataset_after_15.index.year<=2023)]

    dataset_until_15 = dataset_until_15.drop(dataset_until_15.columns[-1], axis=1)

    dataset = pd.concat((dataset_until_15, dataset_after_15))

    dataset = dataset.loc[:, ~dataset.loc[dataset.index.year>=2022].isna().all(axis=0).values]

    imputer = MissForest(criterion="squared_error")
    train_inputed = imputer.fit_transform(dataset.loc[dataset.index.year<2022])
    dataset_train_inputed = pd.DataFrame(train_inputed, columns=dataset.columns, index=dataset.loc[dataset.index.year<2022].index)

    valtest_inputed = imputer.transform(dataset.loc[dataset.index.year>=2022])
    dataset_valtest_inputed = pd.DataFrame(valtest_inputed, columns=dataset.columns,index=dataset.loc[dataset.index.year>=2022].index)

    dataset_inputed = pd.concat((dataset_train_inputed, dataset_valtest_inputed))

    dataset_inputed.columns = [col.split("-")[1].split()[0].lower() for col in dataset_inputed.columns]

    dataset_inputed["year"] = dataset_inputed.index.year

    dataset_inputed = dataset_inputed[["year", *dataset_inputed.columns[:-1]]]

    data_folder = f"../../processed/{file.split('_')[0]}0523"
    if not os.path.isdir(data_folder):
        os.mkdir(data_folder)

    dataset_inputed.to_csv(f"{data_folder}/data.csv")

In [ ]:
for file in glob(f"../../processed/*0523/data.csv"):

    data = pd.read_csv(file)
    o3_series = data["o3"]

    data.drop(["o3"], axis=1, inplace=True)


    data["target_o3"] = o3_series

    data.to_csv(file, index=None)


In [ ]:
data